# Diversity by County

# Clean data
Merge the column with the data dictionary + merge multiracial categories. 

In [7]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
import matplotlib.pyplot as plt

In [21]:
racial_census_df = pd.read_csv('../data/racialcensusdata.csv')

In [22]:
racial_census_df.head(5)

,GEO_ID,NAME,P9_001N,P9_002N,P9_003N,P9_004N,P9_005N,P9_006N,P9_007N,P9_008N,...,P9_065N,P9_066N,P9_067N,P9_068N,P9_069N,P9_070N,P9_071N,P9_072N,P9_073N,Unnamed: 75
0,Geography,Geographic Area Name,!!Total:,!!Total:!!Hispanic or Latino,!!Total:!!Not Hispanic or Latino:,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,!!Total:!!Not Hispanic or Latino:!!Population...,NaN
1,0500000US01001,"Autauga County, Alabama",58805,2117,56688,54198,41582,11352,184,873,...,0,0,0,0,0,0,0,0,0,NaN
2,0500000US01003,"Baldwin County, Alabama",231767,12686,219081,208713,186495,18001,1291,2029,...,0,0,0,0,0,0,0,0,0,NaN
3,0500000US01005,"Barbour County, Alabama",25223,1510,23713,23160,11086,11850,58,103,...,2,2,0,0,0,0,0,0,0,NaN
4,0500000US01007,"Bibb County, Alabama",22293,740,21553,20953,16442,4390,39,26,...,0,0,0,0,0,0,0,0,0,NaN


In [36]:
df_subset = racial_census_df.iloc[:, :13].copy()

In [37]:
df_subset

,Geography,Geographic Area Name,!!Total:,!!Total:!!Hispanic or Latino,!!Total:!!Not Hispanic or Latino:,!!Total:!!Not Hispanic or Latino:!!Population of one race:,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!White alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!Black or African American alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!American Indian and Alaska Native alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!Asian alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!Native Hawaiian and Other Pacific Islander alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!Some Other Race alone,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:
0,0500000US01001,"Autauga County, Alabama",58805,2117,56688,54198,41582,11352,184,873,22,185,2490
1,0500000US01003,"Baldwin County, Alabama",231767,12686,219081,208713,186495,18001,1291,2029,122,775,10368
2,0500000US01005,"Barbour County, Alabama",25223,1510,23713,23160,11086,11850,58,103,0,63,553
3,0500000US01007,"Bibb County, Alabama",22293,740,21553,20953,16442,4390,39,26,9,47,600
4,0500000US01009,"Blount County, Alabama",59134,5771,53363,51063,49764,826,188,174,11,100,2300
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3216,0500000US72145,"Vega Baja Municipio, Puerto Rico",54414,54012,402,353,279,31,9,11,0,23,49
3217,0500000US72147,"Vieques Municipio, Puerto Rico",8249,7586,663,653,533,47,15,3,0,55,10
3218,0500000US72149,"Villalba Municipio, Puerto Rico",22093,22008,85,80,65,3,5,1,0,6,5
3219,0500000US72151,"Yabucoa Municipio, Puerto Rico",30426,30258,168,156,115,12,1,8,2,18,12


In [38]:
# Create a dictionary mapping the current index to a clean name
new_names = {
    df_subset.columns[0]: 'GEO_ID',
    df_subset.columns[1]: 'County_Name',
    df_subset.columns[2]: 'Total_Pop',
    df_subset.columns[3]: 'Hispanic_Latino',
    df_subset.columns[4]: 'Not_Hispanic_Total',
    df_subset.columns[5]: 'One_Race_Total',
    df_subset.columns[6]: 'White_alone',
    df_subset.columns[7]: 'Black_alone',
    df_subset.columns[8]: 'AIAN_alone',   # American Indian / Alaska Native
    df_subset.columns[9]: 'Asian_alone',
    df_subset.columns[10]: 'NHPI_alone',  # Native Hawaiian / Pacific Islander
    df_subset.columns[11]: 'Other_alone',
    df_subset.columns[12]: 'Two_or_More'  # This is the multiracial category
}

# Apply the rename
df_subset = df_subset.rename(columns=new_names)

# Make sure the numbers are actually numbers (int) for analysis
cols_to_fix = df_subset.columns[2:]
df_subset[cols_to_fix] = df_subset[cols_to_fix].apply(pd.to_numeric)

df_subset.head()

,GEO_ID,County_Name,Total_Pop,Hispanic_Latino,Not_Hispanic_Total,One_Race_Total,White_alone,Black_alone,AIAN_alone,Asian_alone,NHPI_alone,Other_alone,Two_or_More
0,0500000US01001,"Autauga County, Alabama",58805,2117,56688,54198,41582,11352,184,873,22,185,2490
1,0500000US01003,"Baldwin County, Alabama",231767,12686,219081,208713,186495,18001,1291,2029,122,775,10368
2,0500000US01005,"Barbour County, Alabama",25223,1510,23713,23160,11086,11850,58,103,0,63,553
3,0500000US01007,"Bibb County, Alabama",22293,740,21553,20953,16442,4390,39,26,9,47,600
4,0500000US01009,"Blount County, Alabama",59134,5771,53363,51063,49764,826,188,174,11,100,2300
